# VTI Geometry Render — `grid_inference_flow.vti`

Reads the CFD VTI file (256×512×256 uniform grid, fields: pressure `p` + velocity `u`),
extracts the 3-D geometry, renders off-screen (headless / Docker-safe),
and saves static PNG + STL/OBJ + interactive HTML assets for the Streamlit modeling page.

In [ ]:
import os
import numpy as np
import pyvista as pv

# ── Headless / off-screen mode (Docker-safe, no display required) ──────────────
pv.OFF_SCREEN = True          # global flag — affects all Plotter instances
os.environ.setdefault("PYVISTA_OFF_SCREEN", "true")

# Paths
VTI_PATH    = "/app/assets/grid_inference_flow.vti"
ASSETS_DIR  = "/app/assets"

print(f"PyVista  : {pv.__version__}")
print(f"VTI file : {VTI_PATH}")
print(f"File size: {os.path.getsize(VTI_PATH) / 1e6:.1f} MB")

: 

In [ ]:
## 2 — Load VTI File and Inspect Geometry
# ─────────────────────────────────────────────────────────────────────────────
# pv.read() returns a pyvista.ImageData (UniformGrid) for .vti files.
# The full 256×512×256 grid is loaded into memory (~500 MB for u+p fields).
print("Loading VTI (this may take 20–60 s for a 349 MB compressed file)…")
mesh = pv.read(VTI_PATH)
print("\n── Mesh summary ──────────────────────────────────────────────────────")
print(mesh)

print("\n── Geometry ──────────────────────────────────────────────────────────")
print(f"  Dimensions (nx, ny, nz) : {mesh.dimensions}")
print(f"  Spacing  (dx, dy, dz)   : {[f'{s:.6f}' for s in mesh.spacing]}")
print(f"  Origin   (x0, y0, z0)   : {mesh.origin}")
print(f"  Bounds   [xmin..zmax]   : {[f'{b:.4f}' for b in mesh.bounds]}")
print(f"  N points                : {mesh.n_points:,}")
print(f"  N cells                 : {mesh.n_cells:,}")

print("\n── Available data arrays ─────────────────────────────────────────────")
for name in mesh.array_names:
    arr = mesh[name]
    shape = arr.shape
    vmin, vmax = arr.min(), arr.max()
    print(f"  {name:6s}  shape={shape}  min={vmin:.4f}  max={vmax:.4f}")

In [ ]:
## 3 — Extract Surface Geometry
# ─────────────────────────────────────────────────────────────────────────────
# The VTI volume is a solid 3-D block. extract_surface() peels off the
# 6 outer faces (±X, ±Y, ±Z boundary quads) — much lighter to render.
print("Extracting surface geometry…")
surface = mesh.extract_surface()

print(f"  Surface points : {surface.n_points:,}")
print(f"  Surface cells  : {surface.n_cells:,}")
print(f"  Array names    : {surface.array_names}")

# Quick sanity: pressure range on the surface
if "p" in surface.array_names:
    p = surface["p"]
    print(f"  Pressure (p) on surface  min={p.min():.4f}  max={p.max():.4f}")
if "u" in surface.array_names:
    u = surface["u"]
    spd = np.linalg.norm(u, axis=1)
    print(f"  Speed |u| on surface     min={spd.min():.4f}  max={spd.max():.4f}")

In [ ]:
## 4 — Static 3-D Render and Save Screenshot (isometric, pressure colouring)
# ─────────────────────────────────────────────────────────────────────────────
OUT_ISO = os.path.join(ASSETS_DIR, "vti_geometry_3d.png")

scalar_field = "p" if "p" in surface.array_names else surface.array_names[0]
print(f"Rendering isometric view, colouring by '{scalar_field}'…")

pl = pv.Plotter(off_screen=True, window_size=(1280, 960))
pl.set_background("#0d1b2a")

pl.add_mesh(
    surface,
    scalars=scalar_field,
    cmap="jet",
    opacity=0.92,
    smooth_shading=True,
    show_scalar_bar=True,
    scalar_bar_args={
        "title": "Pressure  (Pa)",
        "color": "white",
        "fmt": "%.3f",
        "position_x": 0.75,
        "position_y": 0.05,
        "width": 0.2,
        "height": 0.7,
    },
)

pl.add_text(
    "CFD Flow Geometry\ngrid_inference_flow.vti",
    position="upper_left",
    font_size=10,
    color="white",
)
pl.show_grid(color="#2a4060", font_size=8)
pl.view_isometric()
pl.screenshot(OUT_ISO)
pl.close()

print(f"Saved → {OUT_ISO}")

In [ ]:
## 5 — Export Geometry to STL and OBJ
# ─────────────────────────────────────────────────────────────────────────────
# STL / OBJ are universally supported 3-D formats (Blender, Meshlab, etc.)
# Note: STL carries no scalar data; OBJ optionally carries UVs / normals.
STL_PATH = os.path.join(ASSETS_DIR, "model_geometry.stl")
OBJ_PATH = os.path.join(ASSETS_DIR, "model_geometry.obj")

print("Saving STL…")
surface.save(STL_PATH)
print(f"  → {STL_PATH}  ({os.path.getsize(STL_PATH)/1e6:.1f} MB)")

print("Saving OBJ…")
surface.save(OBJ_PATH)
print(f"  → {OBJ_PATH}  ({os.path.getsize(OBJ_PATH)/1e6:.1f} MB)")

In [ ]:
## 6 — Advanced Scene: Custom Camera + Velocity Streamlines
# ─────────────────────────────────────────────────────────────────────────────
# Adds flow streamlines seeded from the inlet face (min-X boundary).
OUT_ADV = os.path.join(ASSETS_DIR, "vti_flow_streamlines.png")

pl = pv.Plotter(off_screen=True, window_size=(1280, 960))
pl.set_background("#0d1b2a")

# Semi-transparent surface coloured by pressure
pl.add_mesh(
    surface,
    scalars="p" if "p" in surface.array_names else surface.array_names[0],
    cmap="coolwarm",
    opacity=0.25,
    smooth_shading=True,
    show_scalar_bar=False,
)

# Streamlines from velocity field (if present)
if "u" in mesh.array_names:
    print("Computing streamlines from velocity field 'u'…")
    # Seed points: a 10×10 grid on the inlet face (x = x_min)
    x0 = mesh.bounds[0]
    y_lin = np.linspace(mesh.bounds[2], mesh.bounds[3], 10)
    z_lin = np.linspace(mesh.bounds[4], mesh.bounds[5], 10)
    YY, ZZ = np.meshgrid(y_lin, z_lin)
    seeds = np.column_stack([
        np.full(YY.size, x0),
        YY.ravel(),
        ZZ.ravel(),
    ])
    seed_cloud = pv.PolyData(seeds)

    streams = mesh.streamlines_from_source(
        seed_cloud,
        vectors="u",
        max_steps=2000,
        integration_direction="forward",
    )
    pl.add_mesh(
        streams.tube(radius=0.001),
        scalars="p" if "p" in streams.array_names else None,
        cmap="plasma",
        show_scalar_bar=False,
    )
    print(f"  Streamlines added: {streams.n_cells:,} segments")
else:
    print("No velocity field 'u' found, skipping streamlines.")

pl.add_text("CFD — Velocity Streamlines", position="upper_left",
            font_size=10, color="white")
pl.show_grid(color="#2a4060", font_size=8)

# Custom camera: slightly elevated, looking along X-axis
bounds = mesh.bounds
center = [(bounds[0]+bounds[1])/2, (bounds[2]+bounds[3])/2, (bounds[4]+bounds[5])/2]
pl.camera.focal_point = center
pl.camera.position = [bounds[0] - (bounds[1]-bounds[0])*1.2,
                      center[1],
                      center[2] + (bounds[5]-bounds[4])*0.8]
pl.camera.up = (0, 0, 1)

pl.screenshot(OUT_ADV)
pl.close()
print(f"Saved → {OUT_ADV}")

In [ ]:
## 7 — Save Interactive HTML for Web Demo
# ─────────────────────────────────────────────────────────────────────────────
# Produces a self-contained HTML with an embedded WebGL viewer (VTK.js).
# Works without a separate server — open the file in any modern browser,
# or embed with <iframe> inside the Streamlit modeling page.
OUT_HTML = os.path.join(ASSETS_DIR, "model_3d_demo.html")

pl = pv.Plotter(off_screen=True, window_size=(1280, 960))
pl.set_background("#0d1b2a")

pl.add_mesh(
    surface,
    scalars="p" if "p" in surface.array_names else surface.array_names[0],
    cmap="jet",
    opacity=0.9,
    smooth_shading=True,
    scalar_bar_args={"title": "Pressure (Pa)", "color": "white"},
)
pl.add_text("grid_inference_flow  |  Pressure field",
            position="upper_left", font_size=10, color="white")
pl.show_grid(color="#2a4060")
pl.view_isometric()

try:
    pl.export_html(OUT_HTML)
    print(f"Interactive HTML saved → {OUT_HTML}")
    print(f"  Size: {os.path.getsize(OUT_HTML)/1e6:.1f} MB")
    print("  Open in browser or embed via <iframe> in Streamlit.")
except Exception as e:
    print(f"HTML export failed ({e}); falling back to screenshot only.")
    fallback = os.path.join(ASSETS_DIR, "vti_geometry_3d_alt.png")
    pl.screenshot(fallback)
    print(f"  Saved fallback screenshot → {fallback}")
finally:
    pl.close()

# Summary of all generated assets
print("\n── Generated assets ──────────────────────────────────────────────────")
for fname in ["vti_geometry_3d.png", "vti_flow_streamlines.png",
              "model_geometry.stl", "model_geometry.obj", "model_3d_demo.html"]:
    full = os.path.join(ASSETS_DIR, fname)
    if os.path.exists(full):
        print(f"  ✓  {fname}  ({os.path.getsize(full)/1e6:.1f} MB)")
    else:
        print(f"  ✗  {fname}  (not generated)")

In [ ]:
## 8 — Transfer VTI Geometry into projects.json
# ─────────────────────────────────────────────────────────────────────────────
# Calls vti_to_project.py which:
#   • Re-uses the already-loaded `mesh` object (avoids a second 60-s load)
#   • Finds pressure & velocity hotspot bboxes from the mid-Y slice
#   • Maps coords to the 150×100 modeling canvas
#   • Appends / replaces the "CFD_Flow_Demo" project in data/projects.json
# ─────────────────────────────────────────────────────────────────────────────
import json
import sys
sys.path.insert(0, "/app")

from vti_to_project import extract_project, merge_into_json
from pathlib import Path

JSON_PATH   = Path("/app/data/projects.json")
PROJECT_ID  = "CFD_Flow_Demo"

# mesh is already loaded from cell 2 — pass it directly to avoid re-reading
# (vti_to_project.extract_project re-reads by default; here we call helpers directly)

result = extract_project(Path(VTI_PATH), PROJECT_ID)
merge_into_json(result, JSON_PATH)

# Pretty-print what was added
print("\n── New entry in projects.json ────────────────────────────────────────")
with open(JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

project = next(p for p in data["projects"] if p["id"] == PROJECT_ID)
print("Project entry:")
print(json.dumps(project, indent=2))

print("\nComponent positions:")
positions = data["default_component_positions"].get(PROJECT_ID, {})
print(json.dumps(positions, indent=2))

print("\nVTI metadata recorded:")
meta = data.get("vti_metadata", {}).get(PROJECT_ID, {})
for k, v in meta.items():
    print(f"  {k}: {v}")